# Sklearn Is Fitted

========================================
`__sklearn_is_fitted__` as Developer API
========================================

The `__sklearn_is_fitted__` method is a convention used in scikit-learn for
checking whether an estimator object has been fitted or not. This method is
typically implemented in custom estimator classes that are built on top of
scikit-learn's base classes like `BaseEstimator` or its subclasses.

Developers should use :func:`~sklearn.utils.validation.check_is_fitted`
at the beginning of all methods except `fit`. If they need to customize or
speed-up the check, they can implement the `__sklearn_is_fitted__` method as
shown below.

In this example the custom estimator showcases the usage of the
`__sklearn_is_fitted__` method and the `check_is_fitted` utility function
as developer APIs. The `__sklearn_is_fitted__` method checks fitted status
by verifying the presence of the `_is_fitted` attribute.

### Imports for Implementing the `__sklearn_is_fitted__` Developer API

**`check_is_fitted` and `__sklearn_is_fitted__` form the contract between scikit-learn's infrastructure and custom estimators for guarding against calling `predict` or `transform` before `fit`**: By default, `check_is_fitted` inspects the estimator for attributes ending with `_` (like `coef_` or `classes_`) that are conventionally set during `fit`. However, custom estimators can override this behavior by implementing `__sklearn_is_fitted__` to return a boolean, enabling faster or more precise fitted-status checks -- for example, checking a single `_is_fitted` flag instead of scanning all attributes.

**This pattern is essential for developing scikit-learn-compatible estimators that integrate with pipelines, cross-validation, and meta-estimators**: When a custom estimator is used inside `Pipeline`, `GridSearchCV`, or `VotingClassifier`, scikit-learn internally calls `check_is_fitted` before routing data through `predict` or `transform`. If the check fails, a `NotFittedError` is raised with a clear message, preventing silent bugs from unfitted models producing meaningless predictions. The `BaseEstimator` and `ClassifierMixin` base classes provide the boilerplate for `get_params`, `set_params`, and `score`, while the developer is responsible for implementing `fit`, `predict`, and the fitted-status protocol.

In [ ]:
# %%
# An example custom estimator implementing a simple classifier
# ------------------------------------------------------------
# This code snippet defines a custom estimator class called `CustomEstimator`
# that extends both the `BaseEstimator` and `ClassifierMixin` classes from
# scikit-learn and showcases the usage of the `__sklearn_is_fitted__` method
# and the `check_is_fitted` utility function.

# Authors: The scikit-learn developers
# SPDX-License-Identifier: BSD-3-Clause

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted

In [ ]:
class CustomEstimator(BaseEstimator, ClassifierMixin):
    def __init__(self, parameter=1):
        self.parameter = parameter

    def fit(self, X, y):
        """
        Fit the estimator to the training data.
        """
        self.classes_ = sorted(set(y))
        # Custom attribute to track if the estimator is fitted
        self._is_fitted = True
        return self

    def predict(self, X):
        """
        Perform Predictions

        If the estimator is not fitted, then raise NotFittedError
        """
        check_is_fitted(self)
        # Perform prediction logic
        predictions = [self.classes_[0]] * len(X)
        return predictions

    def score(self, X, y):
        """
        Calculate Score

        If the estimator is not fitted, then raise NotFittedError
        """
        check_is_fitted(self)
        # Perform scoring logic
        return 0.5

    def __sklearn_is_fitted__(self):
        """
        Check fitted status and return a Boolean value.
        """
        return hasattr(self, "_is_fitted") and self._is_fitted